In [172]:
import pandas as pd
import numpy as np
import re
import ast
import json
import os
import joblib

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, accuracy_score

In [173]:
train_df = pd.read_excel("DeepX_train.xlsx")
val_df = pd.read_excel("DeepX_validation.xlsx")
unlabeled_df = pd.read_excel("DeepX_unlabeled.xlsx")

print(train_df.shape)
print(val_df.shape)
print(unlabeled_df.shape)

display(train_df.head())

(1971, 9)
(500, 9)
(7047, 7)


,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""ne..."
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""posit..."
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]","{""general"": ""positive""}"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]","{""food"": ""positive"", ""price"": ""positive""}"


In [174]:
print(train_df.isna().sum())
print(val_df.isna().sum())
print(unlabeled_df.isna().sum())

print(train_df["platform"].value_counts())
print(train_df["business_category"].value_counts().head(20))

review_id            0
review_text          0
star_rating          0
date                 0
business_name        0
business_category    0
platform             0
aspects              0
aspect_sentiments    0
dtype: int64
review_id            0
review_text          0
star_rating          0
date                 0
business_name        0
business_category    0
platform             0
aspects              0
aspect_sentiments    0
dtype: int64
review_id            0
review_text          0
star_rating          0
date                 0
business_name        0
business_category    0
platform             0
dtype: int64
platform
google_maps    1210
play_store      761
Name: count, dtype: int64
business_category
ecommerce             212
مطعم                  150
transport             129
travel                121
entertainment         117
real_estate           100
عِيادة أسنان           89
food_delivery          82
مستشفى                 79
مطعم مأكولات سورية     77
مطعم مأكولات مشوية     75
صالون ت

In [175]:
def remove_tashkeel(text):
    text = str(text)
    tashkeel = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    return re.sub(tashkeel, '', text)

def normalize_arabic(text):
    text = str(text)
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ة", "ه", text)
    return text

def remove_repeated_chars(text):
    text = str(text)
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def clean_text(text):
    text = str(text)
    text = remove_tashkeel(text)
    text = normalize_arabic(text)
    text = remove_repeated_chars(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [176]:
for df in [train_df, val_df, unlabeled_df]:
    df["clean_text"] = df["review_text"].apply(clean_text)

for df in [train_df, val_df, unlabeled_df]:
    df["model_text"] = (
        df["clean_text"].astype(str) + " " +
        df["business_category"].astype(str) + " " +
        df["platform"].astype(str) + " " +
        df["star_rating"].astype(str)
    )

display(train_df[["review_text", "clean_text", "model_text"]].head())

,review_text,clean_text,model_text
0,لا يوجد الدفع بالبطاقه عند الاستلام,لا يوجد الدفع بالبطاقه عند الاستلام,لا يوجد الدفع بالبطاقه عند الاستلام ecommerce ...
1,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...,المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...
2,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,تجربه سيئه سالتهم الاكل هياخد وقت قد ايه قالول...,تجربه سيئه سالتهم الاكل هياخد وقت قد ايه قالول...
3,احلي مكان فزايد,احلي مكان فزايد,احلي مكان فزايد مطعم مأكولات ومشروبات google_m...
4,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,الفطير حلو جدا الاحجام تحفه بالنسبه للسعر فا ي...,الفطير حلو جدا الاحجام تحفه بالنسبه للسعر فا ي...


In [177]:
ASPECTS = ["food", "service", "price", "cleanliness", "delivery", "ambiance", "app_experience", "general"]

sentiment_to_label = {
    "positive": 1,
    "negative": 2,
    "neutral": 0
}

label_to_sentiment = {
    1: "positive",
    2: "negative"
}

In [178]:
def safe_parse_dict(value):
    try:
        return ast.literal_eval(value) if isinstance(value, str) else value
    except:
        return {}

def create_aspect_labels(row):
    labels = {aspect + "_label": 0 for aspect in ASPECTS}
    sentiments = safe_parse_dict(row["aspect_sentiments"])
    
    for aspect in ASPECTS:
        if aspect in sentiments:
            labels[aspect + "_label"] = sentiment_to_label.get(sentiments[aspect], 0)
    
    return pd.Series(labels)

label_cols = [aspect + "_label" for aspect in ASPECTS]

train_labels = train_df.apply(create_aspect_labels, axis=1)
val_labels = val_df.apply(create_aspect_labels, axis=1)

train_df = pd.concat([train_df.drop(columns=label_cols, errors="ignore"), train_labels], axis=1)
val_df = pd.concat([val_df.drop(columns=label_cols, errors="ignore"), val_labels], axis=1)

display(train_df[label_cols].head())

,food_label,service_label,price_label,cleanliness_label,delivery_label,ambiance_label,app_experience_label,general_label
0,0,0,0,0,2,0,2,0
1,0,1,0,1,0,1,0,0
2,0,2,0,0,2,0,0,0
3,0,0,0,0,0,0,0,1
4,1,0,1,0,0,0,0,0


In [179]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    max_features=50000,
    min_df=2
)

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=50000,
    min_df=2
)

X_train_word = word_vectorizer.fit_transform(train_df["model_text"])
X_val_word = word_vectorizer.transform(val_df["model_text"])

X_train_char = char_vectorizer.fit_transform(train_df["model_text"])
X_val_char = char_vectorizer.transform(val_df["model_text"])

X_train = hstack([X_train_word, X_train_char])
X_val = hstack([X_val_word, X_val_char])

print(X_train.shape)
print(X_val.shape)

(1971, 57523)
(500, 57523)


In [180]:
from sklearn.svm import LinearSVC

models_svm = {}

for aspect in ASPECTS:
    col = aspect + "_label"
    
    model = LinearSVC(
        class_weight="balanced",
        C=1.0,
        max_iter=5000
    )
    
    model.fit(X_train, train_df[col])
    models_svm[aspect] = model
    
    print(f"Trained SVM model for {aspect}")

Trained SVM model for food
Trained SVM model for service
Trained SVM model for price
Trained SVM model for cleanliness
Trained SVM model for delivery
Trained SVM model for ambiance
Trained SVM model for app_experience
Trained SVM model for general


In [181]:
def absa_score(y_true_all, y_pred_all):
    filtered_true = []
    filtered_pred = []

    for t, p in zip(y_true_all, y_pred_all):
        if t != 0 or p != 0:
            filtered_true.append(t)
            filtered_pred.append(p)

    return f1_score(filtered_true, filtered_pred, average="micro", zero_division=0)

In [189]:
from sklearn.metrics import classification_report, accuracy_score

all_true = []
all_pred = []

for aspect in ASPECTS:
    col = aspect + "_label"
    
    y_true = val_df[col]
    y_pred = models_svm[aspect].predict(X_val)

    print("=" * 70)
    print(aspect)
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred, zero_division=0))
    
    all_true.extend(y_true)
    all_pred.extend(y_pred)



food
Accuracy: 0.894
              precision    recall  f1-score   support

           0       0.94      0.95      0.94       407
           1       0.64      0.74      0.68        53
           2       0.79      0.57      0.67        40

    accuracy                           0.89       500
   macro avg       0.79      0.75      0.76       500
weighted avg       0.90      0.89      0.89       500

service
Accuracy: 0.848
              precision    recall  f1-score   support

           0       0.85      0.87      0.86       247
           1       0.90      0.87      0.89       151
           2       0.75      0.75      0.75       102

    accuracy                           0.85       500
   macro avg       0.84      0.83      0.83       500
weighted avg       0.85      0.85      0.85       500

price
Accuracy: 0.912
              precision    recall  f1-score   support

           0       0.91      0.99      0.95       422
           1       0.78      0.39      0.52        18
        

In [183]:
os.makedirs("models", exist_ok=True)

joblib.dump(word_vectorizer, "models/word_vectorizer.pkl")
joblib.dump(char_vectorizer, "models/char_vectorizer.pkl")
joblib.dump(models_svm, "models/models_svm.pkl")

print("Models saved.")

Models saved.


In [184]:
X_test_word = word_vectorizer.transform(unlabeled_df["model_text"])
X_test_char = char_vectorizer.transform(unlabeled_df["model_text"])
X_test = hstack([X_test_word, X_test_char])

print("Test features ready:", X_test.shape)

Test features ready: (7047, 57523)


In [185]:
submission = []

for i, row in unlabeled_df.iterrows():
    review_id = int(row["review_id"])
    
    predicted_aspects = []
    aspect_sentiments = {}
    
    for aspect in ASPECTS:
        pred = models_svm[aspect].predict(X_test[i])[0]
        
        if pred != 0:
            predicted_aspects.append(aspect)
            aspect_sentiments[aspect] = label_to_sentiment[pred]
    
    if len(predicted_aspects) == 0:
        predicted_aspects = ["none"]
        aspect_sentiments = {"none": "neutral"}
    
    submission.append({
        "review_id": review_id,
        "aspects": predicted_aspects,
        "aspect_sentiments": aspect_sentiments
    })

print("Prediction finished")

Prediction finished


In [186]:
with open("submission.json", "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print("submission.json saved")

submission.json saved


In [187]:
def validate_submission(submission, unlabeled_df):
    allowed_aspects = set(ASPECTS + ["none"])
    allowed_sentiments = {"positive", "negative", "neutral"}
    
    errors = []
    
    if len(submission) != len(unlabeled_df):
        errors.append("Row count does not match unlabeled file.")
    
    test_ids = set(unlabeled_df["review_id"].astype(int))
    sub_ids = set(item["review_id"] for item in submission)
    
    if test_ids != sub_ids:
        errors.append("Review IDs do not match.")
    
    for item in submission:
        aspects = item.get("aspects")
        sentiments = item.get("aspect_sentiments")
        
        if not isinstance(aspects, list):
            errors.append(f"aspects is not list for review_id {item.get('review_id')}")
            continue
        
        if not isinstance(sentiments, dict):
            errors.append(f"aspect_sentiments is not dict for review_id {item.get('review_id')}")
            continue
        
        if set(aspects) != set(sentiments.keys()):
            errors.append(f"Keys mismatch for review_id {item.get('review_id')}")
        
        if "none" in aspects and len(aspects) > 1:
            errors.append(f"'none' mixed with other aspects for review_id {item.get('review_id')}")
        
        for asp in aspects:
            if asp not in allowed_aspects:
                errors.append(f"Invalid aspect {asp}")
        
        for sent in sentiments.values():
            if sent not in allowed_sentiments:
                errors.append(f"Invalid sentiment {sent}")
    
    if errors:
        print("Submission has errors:")
        for e in errors[:20]:
            print(e)
    else:
        print("Submission is valid")


validate_submission(submission, unlabeled_df)

with open("submission.json", "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print("submission.json saved")
print(len(submission))
print(len(unlabeled_df))
submission[:5]

Submission is valid


submission.json saved
7047
7047


[{'review_id': 1,
  'aspects': ['none'],
  'aspect_sentiments': {'none': 'neutral'}},
 {'review_id': 2,
  'aspects': ['ambiance'],
  'aspect_sentiments': {'ambiance': 'negative'}},
 {'review_id': 3,
  'aspects': ['none'],
  'aspect_sentiments': {'none': 'neutral'}},
 {'review_id': 4,
  'aspects': ['none'],
  'aspect_sentiments': {'none': 'neutral'}},
 {'review_id': 5,
  'aspects': ['general'],
  'aspect_sentiments': {'general': 'positive'}}]

In [188]:
from collections import Counter

aspect_counter = Counter()

for item in submission:
    for asp in item["aspects"]:
        aspect_counter[asp] += 1

aspect_counter

Counter({'general': 2092,
         'service': 1954,
         'none': 1883,
         'app_experience': 681,
         'food': 579,
         'ambiance': 498,
         'price': 261,
         'cleanliness': 124,
         'delivery': 86})